# Structured Output 完全指南

**前置知识**: Python 基础、JSON 格式、Pydantic 基础、上两节内容

**学习目标**: 掌握如何解析和验证 LLM 的结构化输出

---

## 核心问题：LLM 输出不可控

LLM 返回的是自然语言文本，格式不固定：

```
非结构化输出（难以解析）:
"用户名是张三，年龄25岁，住在北京"

结构化输出（易于解析）:
{"name": "张三", "age": 25, "city": "北京"}
```

**StructuredOutputParser 解决方案**：从 LLM 输出中提取 JSON，验证格式，转换为类型安全的对象。

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import sys
import json
sys.path.insert(0, '..')  # 添加父目录到路径

from pydantic import BaseModel, Field  # Pydantic: 数据验证库
from typing import List, Optional

from src.structured_output import (
    StructuredOutputParser,    # 核心解析器
    OutputSchema,              # 输出模式定义
    ValidationError,           # 验证错误
    create_choice_parser,      # 选择解析器工厂
    create_extraction_parser,  # 提取解析器工厂
)

print("导入成功！")

---

## 第一步：使用 Pydantic 定义输出格式

Pydantic 模型 = 类型安全的数据结构

| 特性 | 说明 |
|------|------|
| 类型检查 | 自动验证字段类型 |
| 默认值 | 支持可选字段 |
| 描述 | Field(description=...) 帮助 LLM 理解 |

In [ ]:
# ============================================================
# 定义输出模型
# ============================================================

# 使用 Pydantic 定义期望的输出结构
class Person(BaseModel):
    """人员信息模型"""
    name: str = Field(description="姓名")           # 必需字段
    age: int = Field(description="年龄")            # 必需字段，整数类型
    email: Optional[str] = Field(                   # 可选字段
        default=None, 
        description="邮箱地址"
    )

# 创建解析器（传入模型类）
parser = StructuredOutputParser(Person)

# 模拟 LLM 输出（包含 JSON 代码块）
llm_output = '''
根据用户信息，我提取了以下数据：

```json
{"name": "张三", "age": 25, "email": "zhang@example.com"}
```
'''

# 解析 → 返回 Person 对象（类型安全）
result = parser.parse(llm_output)

print(f"类型: {type(result).__name__}")  # Person
print(f"姓名: {result.name}")            # 张三
print(f"年龄: {result.age}")             # 25
print(f"邮箱: {result.email}")           # zhang@example.com

---

## 第二步：使用 JSON Schema 定义

不想用 Pydantic？可以直接用 JSON Schema 字典。

**适用场景**：动态生成 schema、与其他系统集成

In [ ]:
# ============================================================
# JSON Schema 方式
# ============================================================

# 直接定义 JSON Schema
schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "文章标题"},
        "summary": {"type": "string", "description": "文章摘要"},
        "keywords": {
            "type": "array", 
            "items": {"type": "string"}, 
            "description": "关键词列表"
        },
    },
    "required": ["title", "summary"],  # 必需字段
}

# 创建解析器（传入 schema 字典）
parser = StructuredOutputParser(schema)

llm_output = '''
```json
{
    "title": "Python 入门指南",
    "summary": "本文介绍 Python 编程语言的基础知识",
    "keywords": ["Python", "编程", "入门"]
}
```
'''

# 解析 → 返回字典（非 Pydantic 对象）
result = parser.parse(llm_output)

print(f"标题: {result['title']}")
print(f"摘要: {result['summary']}")
print(f"关键词: {result['keywords']}")

---

## 第三步：生成格式说明

`get_format_instructions()` 生成提示词，告诉 LLM 如何输出。

In [ ]:
# ============================================================
# 生成格式说明（用于 Prompt）
# ============================================================

class TaskResult(BaseModel):
    """任务执行结果"""
    success: bool = Field(description="是否成功")
    message: str = Field(description="结果消息")
    data: Optional[dict] = Field(default=None, description="附加数据")

parser = StructuredOutputParser(TaskResult)

# 获取格式说明 → 添加到 system prompt 中
instructions = parser.get_format_instructions()
print("格式说明 (添加到 prompt 中):")
print(instructions)

---

## 第四步：验证和错误处理

| 模式 | 行为 |
|------|------|
| `strict=True` | 验证失败抛出 ValidationError |
| `strict=False` | 验证失败返回原始数据 |

In [ ]:
# ============================================================
# 严格模式验证
# ============================================================

class StrictModel(BaseModel):
    required_field: str   # 必需
    number_field: int     # 必需，整数类型

# 严格模式（默认）
strict_parser = StructuredOutputParser(StrictModel, strict=True)

# 测试1：缺少必需字段
try:
    strict_parser.parse('{"number_field": 42}')
except ValidationError as e:
    print(f"缺少字段: {e}")

# 测试2：类型错误
try:
    strict_parser.parse('{"required_field": "test", "number_field": "not_a_number"}')
except ValidationError as e:
    print(f"类型错误: {e}")

---

## 第五步：自动修复 JSON 错误

LLM 常犯的 JSON 格式错误：

| 错误类型 | 示例 | 修复后 |
|----------|------|--------|
| 尾随逗号 | `{"a": 1,}` | `{"a": 1}` |
| 单引号 | `{'a': 1}` | `{"a": 1}` |
| Python 布尔 | `True` | `true` |
| Python None | `None` | `null` |

In [ ]:
# ============================================================
# 自动修复模式
# ============================================================

class SimpleModel(BaseModel):
    value: str

# 启用自动修复
auto_fix_parser = StructuredOutputParser(SimpleModel, auto_fix=True)

# 测试：尾随逗号（常见错误）
result = auto_fix_parser.parse('{"value": "test",}')  # 注意末尾的逗号
print(f"修复尾随逗号: {result.value}")

# 测试：Python 布尔值
class BoolModel(BaseModel):
    flag: bool

bool_parser = StructuredOutputParser(BoolModel, auto_fix=True)
# True/False 是 Python 格式，不是 JSON 格式
# auto_fix 会将 True → true, False → false

---

## 第六步：安全解析 parse_or_none

不想处理异常？用 `parse_or_none()`，失败返回 None。

In [ ]:
# ============================================================
# 安全解析（不抛异常）
# ============================================================

class Data(BaseModel):
    x: int

parser = StructuredOutputParser(Data)

# 有效输入 → 返回对象
result = parser.parse_or_none('{"x": 42}')
print(f"有效输入: {result}")

# 无效输入 → 返回 None（不抛异常）
result = parser.parse_or_none('invalid json')
print(f"无效输入: {result}")

# 配合 walrus 运算符使用
if result := parser.parse_or_none('{"x": 100}'):
    print(f"解析成功: x = {result.x}")
else:
    print("解析失败")

---

## 第七步：预定义解析器

常用场景的快捷工厂函数：

| 工厂函数 | 用途 |
|----------|------|
| `create_choice_parser()` | 让 LLM 从选项中选择 |
| `create_extraction_parser()` | 从文本提取指定字段 |

In [ ]:
# ============================================================
# 选择解析器
# ============================================================

# 创建选择解析器（限定选项）
choice_parser = create_choice_parser(["approve", "reject", "pending"])

llm_output = '''
经过审核，我的决定是：
```json
{"choice": "approve", "reason": "申请材料完整，符合要求"}
```
'''

result = choice_parser.parse(llm_output)
print(f"选择: {result['choice']}")
print(f"原因: {result['reason']}")

In [ ]:
# ============================================================
# 信息提取解析器
# ============================================================

# 创建提取解析器（指定要提取的字段）
extraction_parser = create_extraction_parser({
    "product_name": "产品名称",
    "price": "价格",
    "category": "产品类别",
})

llm_output = '''
从商品描述中提取的信息：
```json
{
    "product_name": "iPhone 15 Pro",
    "price": "7999元",
    "category": "智能手机"
}
```
'''

result = extraction_parser.parse(llm_output)
print(f"产品: {result['product_name']}")
print(f"价格: {result['price']}")
print(f"类别: {result['category']}")

---

## 第八步：复杂嵌套结构

Pydantic 支持嵌套模型，处理复杂数据结构。

In [ ]:
# ============================================================
# 嵌套模型
# ============================================================

# 定义嵌套结构
class Address(BaseModel):
    city: str
    street: str
    zip_code: str

class Company(BaseModel):
    name: str
    industry: str

class Employee(BaseModel):
    name: str
    age: int
    address: Address       # 嵌套对象
    company: Company       # 嵌套对象
    skills: List[str]      # 列表

parser = StructuredOutputParser(Employee)

llm_output = '''
```json
{
    "name": "李四",
    "age": 30,
    "address": {
        "city": "北京",
        "street": "中关村大街1号",
        "zip_code": "100080"
    },
    "company": {
        "name": "科技公司",
        "industry": "互联网"
    },
    "skills": ["Python", "机器学习", "数据分析"]
}
```
'''

result = parser.parse(llm_output)
print(f"姓名: {result.name}")
print(f"城市: {result.address.city}")      # 访问嵌套属性
print(f"公司: {result.company.name}")
print(f"技能: {result.skills}")

---

## 实战：信息提取 Agent

In [ ]:
# ============================================================
# 完整示例：实体关系提取
# ============================================================

# 定义实体
class Entity(BaseModel):
    name: str = Field(description="实体名称")
    type: str = Field(description="实体类型 (人物/地点/组织/时间)")

# 定义关系
class Relation(BaseModel):
    subject: str = Field(description="主体")
    predicate: str = Field(description="关系类型")
    object: str = Field(description="客体")

# 定义提取结果
class ExtractionResult(BaseModel):
    entities: List[Entity] = Field(description="实体列表")
    relations: List[Relation] = Field(description="关系列表")
    summary: str = Field(description="文本摘要")

# 创建解析器
extraction_parser = StructuredOutputParser(ExtractionResult)

# 模拟 LLM 输出
llm_output = '''
我已经分析了文本，以下是提取结果：

```json
{
    "entities": [
        {"name": "马云", "type": "人物"},
        {"name": "阿里巴巴", "type": "组织"},
        {"name": "杭州", "type": "地点"},
        {"name": "1999年", "type": "时间"}
    ],
    "relations": [
        {"subject": "马云", "predicate": "创立", "object": "阿里巴巴"},
        {"subject": "阿里巴巴", "predicate": "总部位于", "object": "杭州"}
    ],
    "summary": "马云于1999年在杭州创立了阿里巴巴公司"
}
```
'''

result = extraction_parser.parse(llm_output)

print("提取的实体:")
for entity in result.entities:
    print(f"  - {entity.name} ({entity.type})")

print("\n提取的关系:")
for rel in result.relations:
    print(f"  - {rel.subject} --[{rel.predicate}]--> {rel.object}")

print(f"\n摘要: {result.summary}")

---

## 练习

定义 `MovieReview` 模型，包含 `title`, `rating`, `pros`, `cons` 字段。

In [ ]:
# ============================================================
# 练习：电影评论模型
# ============================================================

# TODO: 补全以下代码
class MovieReview(BaseModel):
    """电影评论"""
    # title: str = Field(description="电影名称")
    # rating: float = Field(description="评分 (0-10)")
    # pros: List[str] = Field(description="优点列表")
    # cons: List[str] = Field(description="缺点列表")
    pass

# TODO: 创建解析器并测试
# parser = StructuredOutputParser(MovieReview)
# instructions = parser.get_format_instructions()
# print(instructions)

---

## 本节要点

| 概念 | 作用 | 关键方法 |
|------|------|----------|
| `StructuredOutputParser` | 解析验证 LLM 输出 | `parse()`, `parse_or_none()` |
| Pydantic 模型 | 类型安全的输出定义 | `BaseModel`, `Field` |
| JSON Schema | 灵活的模式定义 | 字典格式 |
| 格式说明 | 生成 prompt 指令 | `get_format_instructions()` |
| 自动修复 | 修复常见 JSON 错误 | `auto_fix=True` |
| 预定义解析器 | 常用场景快捷方式 | `create_choice_parser()` |

**下一步**: 学习 Tool Executor，安全执行工具调用。